# Advanced NN: MNIST Classification with Regularization

In this exercise we explore **regularization techniques** for neural networks using the MNIST handwritten digit dataset.

We will build four variants of a Multi-Layer Perceptron (MLP) and compare their performance:
1. **Baseline** -- no regularization
2. **Dropout** -- randomly zero out activations during training
3. **Batch Normalization** -- normalize activations between layers
4. **L2 Regularization** (weight decay) -- penalize large weights

**Dataset:** MNIST (28x28 greyscale images of digits 0-9, downloaded once via `torchvision` into `./data`; set the `COURSE_DATA` environment variable to use a copy prepared before class).

**Evaluation protocol:** 10,000 of the 60,000 training images are held out as a **validation set**. It is used for per-epoch monitoring, for choosing each model's checkpoint (the epoch with the lowest validation loss), and for comparing the four models. The 10,000 test images are used **once**, at the end, for the model chosen on validation.

**Runtime:** 4 models × 10 epochs on 50,000 training images, about 3 minutes on a recent laptop CPU and up to about 10 minutes on an older one. If time is short, set `NUM_EPOCHS = 5` in section 3.

**Learning objectives:**
1. Understand and implement common regularization strategies.
2. Compare training dynamics and generalization across variants on validation data.
3. Visualize misclassified examples to gain intuition.

---
## 1. Imports

In [ ]:
import copy
import os

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
import torchvision
import torchvision.transforms as transforms

from sklearn.metrics import classification_report

SEED = 42
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__}, device: {DEVICE}")

---
## 2. Load MNIST, Hold Out a Validation Set, and Visualize Samples

We split the 60,000 training images into 50,000 for training and 10,000 for validation with a seeded `random_split`. The validation images drive every decision in this notebook (checkpoint epoch, model comparison); the test images stay untouched until section 9. `download=True` only downloads when the files are missing, so the notebook also runs offline once `DATA_ROOT` holds MNIST.

The normalization constants (0.1307, 0.3081) are the published mean and standard deviation of the MNIST training images; the test images play no part in them (recomputed on our 50,000 training images they agree to three decimals).

**Tasks:**
- Use `torchvision.datasets.MNIST` with `ToTensor()` and `Normalize((0.1307,), (0.3081,))`, and `root=DATA_ROOT` where `DATA_ROOT = os.environ.get("COURSE_DATA", "./data")`
- Split the training images into 50,000 training and 10,000 validation images with `random_split` and a seeded generator
- Create `DataLoader` objects: training with `batch_size=128, shuffle=True`; validation and test without shuffling

In [ ]:
# YOUR CODE HERE
# Hint:
#   transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
#   DATA_ROOT = os.environ.get("COURSE_DATA", "./data")
#   full_train   = torchvision.datasets.MNIST(root=DATA_ROOT, train=True, download=True, transform=transform)
#   test_dataset = torchvision.datasets.MNIST(root=DATA_ROOT, train=False, download=True, transform=transform)
#   train_dataset, val_dataset = random_split(full_train, [50_000, 10_000],
#                                             generator=torch.Generator().manual_seed(SEED))
#   train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
#   val_loader   = DataLoader(val_dataset, batch_size=1000, shuffle=False)
#   test_loader  = DataLoader(test_dataset, batch_size=1000, shuffle=False)   # used once, section 9

In [ ]:
# YOUR CODE HERE
# Hint: plot a 2x8 grid of sample digits using train_dataset[i]


---
## 3. Helper Functions: Train and Evaluate

`train_model` trains for a fixed budget of `NUM_EPOCHS`, evaluates the **validation** set after every epoch, keeps a copy of the weights from the epoch with the lowest validation loss, and restores that checkpoint at the end. The test set is never used during training.

Write three functions:
- `evaluate_loss_acc(model, loader, criterion)` -- mean loss and accuracy in eval mode.
- `train_model(model, train_loader, val_loader, optimizer, criterion, num_epochs=NUM_EPOCHS)` -- returns a history dict (train/val loss and accuracy per epoch, plus `best_epoch`) and leaves the model at its best-validation checkpoint.
- `evaluate_model(model, loader)` -- returns `(y_true, y_pred)` arrays.

**Important:** Flatten the 28x28 images to 784 with `x.view(x.size(0), -1)`, and copy the checkpoint with `copy.deepcopy(model.state_dict())`.

In [ ]:
NUM_EPOCHS = 10   # per model; set to 5 if time is short

# YOUR CODE HERE
# Hint: in train_model, loop over epochs, for each epoch:
#   model.train() -> loop over train_loader -> compute loss, backprop
#   val_loss, val_acc = evaluate_loss_acc(model, val_loader, criterion)   # validation, never test
#   if val_loss < best_loss: keep best_epoch and best_state = copy.deepcopy(model.state_dict())
# After the loop: model.load_state_dict(best_state); history["best_epoch"] = best_epoch
#
# In evaluate_model, model.eval() -> loop over the loader -> collect all predictions

---
## 4. Baseline MLP (No Regularization)

Architecture: Linear(784->256) -> ReLU -> Linear(256->128) -> ReLU -> Linear(128->10)

Use `nn.CrossEntropyLoss()` and `optim.Adam` with `lr=1e-3`. Call `torch.manual_seed(SEED)` before building each model so that every run is reproducible.

In [ ]:
# YOUR CODE HERE
# Hint: define class BaselineMLP(nn.Module) with nn.Sequential
# Then: torch.manual_seed(SEED)
#       baseline_model = BaselineMLP().to(DEVICE)
#       baseline_opt = optim.Adam(baseline_model.parameters(), lr=1e-3)
#       criterion = nn.CrossEntropyLoss()
#       baseline_hist = train_model(baseline_model, train_loader, val_loader, baseline_opt, criterion)

---
## 5. MLP with Dropout

Same architecture as baseline but add `nn.Dropout(0.3)` after each ReLU.

In [ ]:
# YOUR CODE HERE
# Hint: define class DropoutMLP(nn.Module) -- same as Baseline but with nn.Dropout layers
# Train it the same way (torch.manual_seed(SEED) first, validation loader for monitoring)

---
## 6. MLP with Batch Normalization

Same architecture as baseline but add `nn.BatchNorm1d(n)` **after** each Linear layer and **before** ReLU.

In [ ]:
# YOUR CODE HERE
# Hint: define class BatchNormMLP(nn.Module)
# Order: Linear -> BatchNorm1d -> ReLU


---
## 7. MLP with L2 Regularization (Weight Decay)

Use the same `BaselineMLP` architecture but add `weight_decay=1e-4` to the Adam optimizer. (Adam's `weight_decay` is the coupled L2 form; comparing `optim.AdamW` is an extension.)

In [ ]:
# YOUR CODE HERE
# Hint: torch.manual_seed(SEED)
#       l2_model = BaselineMLP().to(DEVICE)
#       l2_opt = optim.Adam(l2_model.parameters(), lr=1e-3, weight_decay=1e-4)
#       l2_hist = train_model(l2_model, train_loader, val_loader, l2_opt, criterion)

---
## 8. Comparison Table and Chart (validation set)

We compare the four models at their restored checkpoints on the **validation** set and choose the one with the highest validation accuracy. The training accuracy in the table is re-measured in eval mode on the 50,000 training images: the per-epoch training accuracy is a running average in train mode (dropout active, weights still changing), which would understate the gap for the dropout model.

The four models usually land close together. Before ranking them, repeat the notebook with a second seed (change `SEED`): a difference smaller than the seed-to-seed change is not evidence.

**Tasks:**
- For each model, compute training accuracy (eval mode, training images) and validation loss and accuracy with `evaluate_loss_acc`
- Build a DataFrame with the selected epoch, training accuracy, validation accuracy, validation loss, and the generalization gap
- Choose the model with the highest validation accuracy (`best_name`, `best_model`)
- Plot validation loss and accuracy curves for all models on the same axes

In [ ]:
# YOUR CODE HERE
# Hint: models = {"Baseline": (baseline_model, baseline_hist), ...}
#       train_eval_loader = DataLoader(train_dataset, batch_size=1000, shuffle=False)
#       columns: Model, Selected Epoch, Train Acc, Val Acc, Val Loss, Generalization Gap

In [ ]:
# YOUR CODE HERE
# Hint: create 1x2 subplots, plot val_loss and val_acc for each model

---
## 9. Final Test Evaluation and Misclassified Examples

The model chosen on validation is now evaluated **once** on the 10,000 test images: this is the only use of the test set. We then look at some of the test images it got wrong. Do not go back and change the model because of this result.

**Tasks:**
- Run `evaluate_model(best_model, test_loader)` once, print the test accuracy, and find the misclassified indices
- Display a 2x8 grid showing the image, true label, and predicted label
- Print the classification report

In [ ]:
# YOUR CODE HERE
# Hint:
#   y_true, y_pred = evaluate_model(best_model, test_loader)   # the only use of the test set
#   misclassified_idx = np.where(y_true != y_pred)[0]
#   Plot a 2x8 grid showing the image, true label, and predicted label

In [ ]:
# YOUR CODE HERE
# Hint: print(classification_report(y_true, y_pred, digits=4))


---
## 10. Summary

In this notebook we:

1. Loaded MNIST via `torchvision`, held out 10,000 training images for validation, and visualized sample digits.
2. Built a **baseline** MLP with no regularization.
3. Added **Dropout** -- randomly zeros activations to reduce co-adaptation.
4. Added **Batch Normalization** -- stabilizes and often accelerates training.
5. Added **L2 regularization** via `weight_decay` in the optimizer.
6. Compared all four variants on validation accuracy and generalization gap, each at its best-validation checkpoint.
7. Evaluated the chosen model once on the test set and visualized misclassified examples.

**Key takeaways:**
- Regularization helps close the gap between training and validation performance.
- BatchNorm often speeds up training; dropout and weight decay target the train-validation gap -- measure whether they narrow it here.
- On MNIST the variants differ by small margins: check a difference against a second seed before ranking.
- Validation data choose the checkpoint and the model; the test set is used once, for the final report.